In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

root = Path.cwd()
if not (root / "data").exists():
    root = root.parent

train_df = pd.read_csv(root / "data/processed/train_split.csv")
val_df = pd.read_csv(root / "data/processed/val_split.csv")

# Select sensors using training data only
all_sensors = [f"sensor_{i}" for i in range(1, 22)]

constant_sensors = [
    s for s in all_sensors
    if train_df[s].nunique() == 1
]

selected_sensors = [
    s for s in all_sensors
    if s not in constant_sensors
]

print("Removed sensors:", constant_sensors)
print("Selected sensors:", len(selected_sensors))


Removed sensors: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Selected sensors: 15


In [2]:

WINDOW = 30

def create_features(data, sensors):
    data = data.sort_values(
        ["engine_id", "cycle"]
    ).reset_index(drop=True)

    grouped = data.groupby("engine_id")

    features = data[["engine_id", "cycle", "RUL"]].copy()

    for sensor in sensors:
        group_sensor = grouped[sensor]

        # Current reading
        features[f"{sensor}_latest"] = data[sensor]

        # Average over last 30 cycles
        features[f"{sensor}_mean30"] = group_sensor.transform(
            lambda x: x.rolling(WINDOW, min_periods=WINDOW).mean()
        )

        # Variation in the last 30 cycles
        features[f"{sensor}_std30"] = group_sensor.transform(
            lambda x: x.rolling(WINDOW, min_periods=WINDOW).std()
        )

        # Change from first to last cycle in window
        features[f"{sensor}_delta30"] = (
            data[sensor] - group_sensor.shift(WINDOW - 1)
        )

    # Keep only complete 30-cycle histories
    return features.dropna().reset_index(drop=True)


train_features = create_features(train_df, selected_sensors)
val_features = create_features(val_df, selected_sensors)

print("=== FEATURE ENGINEERING COMPLETE ===")
print("Training shape:", train_features.shape)
print("Validation shape:", val_features.shape)
print("Sensor features:", len(selected_sensors) * 4)

display(train_features.head())

# Save features
train_features.to_csv(
    root / "data/processed/train_features.csv", index=False
)

val_features.to_csv(
    root / "data/processed/val_features.csv", index=False
)

print("Feature datasets saved!")


=== FEATURE ENGINEERING COMPLETE ===
Training shape: (14241, 63)
Validation shape: (3490, 63)
Sensor features: 60


,engine_id,cycle,RUL,sensor_2_latest,sensor_2_mean30,sensor_2_std30,sensor_2_delta30,sensor_3_latest,sensor_3_mean30,sensor_3_std30,...,sensor_17_std30,sensor_17_delta30,sensor_20_latest,sensor_20_mean30,sensor_20_std30,sensor_20_delta30,sensor_21_latest,sensor_21_mean30,sensor_21_std30,sensor_21_delta30
0,2,30,257,641.78,641.954000,0.278451,-0.11,1590.32,1583.441000,4.028298,...,0.773854,0.0,39.10,39.058000,0.097429,0.16,23.3985,23.438783,0.054306,-0.0600
1,2,31,256,641.97,641.956667,0.278200,0.15,1579.86,1583.308333,4.079911,...,0.773854,-1.0,39.17,39.065667,0.096871,0.11,23.4154,23.437347,0.054336,0.0069
2,2,32,255,641.58,641.948667,0.285618,0.03,1584.52,1583.224000,4.025690,...,0.791478,-1.0,38.97,39.062667,0.098434,-0.14,23.4006,23.437083,0.054500,-0.0244
3,2,33,254,641.79,641.956667,0.277306,0.11,1585.17,1583.119000,3.928087,...,0.805156,1.0,39.01,39.059333,0.098469,-0.12,23.4434,23.437697,0.054462,-0.0593
4,2,34,253,642.12,641.971333,0.273782,0.39,1584.08,1583.116667,3.927475,...,0.833908,0.0,38.87,39.050667,0.103355,-0.31,23.4571,23.436177,0.053208,0.0337


Feature datasets saved!
